*0.4 Deep learning basics*

# nn.Module

**The situation.** A sentiment model is written as loose tensors and functions: weights in one dict, the forward pass in a script, saving handled by hand. It works until a second person needs to load it, move it to a GPU, or list its parameters for the optimizer — and each of those is custom code that breaks.

**nn.Module.** The base class for every model, layer and loss in PyTorch. Subclass it, create layers in `__init__`, write `forward`. In return: `.parameters()` for the optimizer, `.to(device)` for the GPU, `.state_dict()` for saving, `.train()`/`.eval()` for mode switching, and nesting — a module of modules. Hugging Face models are `nn.Module`s underneath.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch
import torch.nn.functional as F
from torch import nn


class SentimentClassifier(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_size: int = 64, classes: int = 2):
        super().__init__()
        self.embedding = nn.EmbeddingBag(
            vocabulary_size, embedding_size, mode="mean"
        )  # average of the word vectors
        self.hidden = nn.Linear(embedding_size, 64)
        self.output = nn.Linear(64, classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        sentence = self.embedding(token_ids)  # (batch, embedding_size)
        sentence = self.dropout(F.relu(self.hidden(sentence)))
        return self.output(sentence)  # raw scores, one per class


model = SentimentClassifier(vocabulary_size=30_522)
parameter_count = 0
for name, parameter in model.named_parameters():
    parameter_count += parameter.numel()
    print(f"{name:<22} {tuple(parameter.shape)}")
print("total parameters:", f"{parameter_count:,}")

batch = torch.randint(0, 30_522, (4, 16))  # 4 sentences of 16 token ids
print("output shape:", tuple(model(batch).shape), "→ (4 sentences, 2 class scores)")
assert tuple(model(batch).shape) == (4, 2)

embedding.weight       (30522, 64)
hidden.weight          (64, 64)
hidden.bias            (64,)
output.weight          (2, 64)
output.bias            (2,)
total parameters: 1,957,698
output shape: (4, 2) → (4 sentences, 2 class scores)


**Reading the output.** Every weight is listed by name and shape, found automatically because the layers were assigned in `__init__`. The embedding table is almost all of the ~2M parameters. Four sentences in, four pairs of scores out.

**What the base class gives you for free.** Saving, loading, and the train/eval switch.

In [3]:
import io

checkpoint = io.BytesIO()
torch.save(
    model.state_dict(), checkpoint
)  # a plain dict of name → tensor; in production this is a file
print("checkpoint size:", round(checkpoint.tell() / 1e6, 1), "MB")

restored = SentimentClassifier(vocabulary_size=30_522)
checkpoint.seek(0)
restored.load_state_dict(torch.load(checkpoint))

model.eval()
restored.eval()  # eval: dropout off, so outputs are repeatable
with torch.no_grad():
    same = torch.allclose(model(batch), restored(batch))
print("restored model gives identical outputs:", same)

model.train()
with torch.no_grad():
    differs = not torch.allclose(model(batch), model(batch))
print("in train mode two calls differ (dropout is active):", differs)
assert same and differs

checkpoint size: 7.8 MB
restored model gives identical outputs: True
in train mode two calls differ (dropout is active): True


**The rule to remember.** Every model is an `nn.Module`: layers in `__init__`, computation in `forward`, nothing else by hand. Save `state_dict()`, not the object.

| Use it when | Don't when | Instead use |
|---|---|---|
| any model or reusable layer | a one-off tensor computation with no parameters | a plain function |

**Watch out**
- Layers stored in a Python list are invisible to `.parameters()`; use `nn.ModuleList`.
- `model.eval()` before inference, always — dropout and batch-norm behave differently in train mode (see Break → Fix).
- `torch.save(model)` pickles the class path; a refactor breaks loading. Save the `state_dict`.